# Extracción de Comentarios Políticos de YouTube a CSV

Este proyecto tiene como objetivo la extracción de comentarios políticos de YouTube de canales específicos (La República y Diario El Comercio) utilizando la YouTube Data API v3, para un posible análisis de desinformación política. Los resultados se guardan en archivos CSV.

In [ ]:
from __future__ import annotations

from dataclasses import dataclass, field
from typing import Any, Dict, Iterable, List, Optional, Tuple
from datetime import datetime
from pathlib import Path
import hashlib
import os
import re
import time
import unicodedata

import pandas as pd
from googleapiclient.discovery import build
from googleapiclient.errors import HttpError

try:
    from rich.console import Console
    from rich.panel import Panel
    from rich.progress import (
        Progress,
        SpinnerColumn,
        BarColumn,
        TextColumn,
        TimeElapsedColumn,
    )
    from rich.table import Table

    RICH_DISPONIBLE = True
except Exception:
    Console = None
    Progress = None
    Panel = None
    Table = None
    SpinnerColumn = BarColumn = TextColumn = TimeElapsedColumn = None
    RICH_DISPONIBLE = False

## 1. Configuración General

Esta sección define la configuración principal para el scraper, incluyendo la API Key de YouTube, los canales a monitorear, las metas de extracción de videos y comentarios, y parámetros para la depuración y filtrado. Es crucial configurar la `API_KEY` y los `CanalFuente`.

In [ ]:
API_KEY = input("Añade tu key: ")

@dataclass
class CanalFuente:
    medio: str
    url_canal: str

@dataclass
class ScraperConfig:
    api_key: str = field(default_factory=lambda: os.getenv("YOUTUBE_API_KEY", API_KEY).strip())

    fuentes_canales: List[CanalFuente] = field(default_factory=lambda: [
        CanalFuente(
            medio="La República",
            url_canal="https://youtube.com/@larepublica?si=pFtcq7fOpUDx2ejt",
        ),
        CanalFuente(
            medio="Diario El Comercio",
            url_canal="https://youtube.com/@diarioelcomercio?si=r_Z1lefwjQo0Qv47",
        ),
    ])

    objetivo_videos: int = 10
    objetivo_comentarios_por_video: int = 50

    max_candidatos_recientes_por_canal: int = 180
    max_videos_politicos_a_intentar: int = 90

    max_comentarios_crudos_por_video: int = 180
    max_resultados_por_pagina: int = 100

    pausa_entre_paginas: float = 0.35
    pausa_entre_videos: float = 0.55
    pausa_entre_canales: float = 0.8

    min_caracteres_comentario: int = 5
    orden_comentarios: str = "relevance"
    puntaje_minimo_politico: int = 9

    filtro_meramente_politico: bool = True
    bloquear_noticias_desastre: bool = True

    carpeta_salida: Path = Path("data/raw")
    archivo_data: str = "comentarios_youtube_politica_depurado.csv"
    archivo_resumen: str = "resumen_scrapeo_youtube_politica.csv"

    @property
    def meta_total_comentarios(self) -> int:
        return self.objetivo_videos * self.objetivo_comentarios_por_video

    @property
    def ruta_data(self) -> Path:
        return self.carpeta_salida / self.archivo_data

    @property
    def ruta_resumen(self) -> Path:
        return self.carpeta_salida / self.archivo_resumen

    def validar(self) -> None:
        if not self.api_key:
            raise ValueError(
                "No se encontró API key. Define la variable de entorno YOUTUBE_API_KEY "
                "o pega temporalmente tu clave en API_KEY. No subas esa clave a GitHub."
            )
        if self.orden_comentarios not in {"relevance", "time"}:
            raise ValueError("orden_comentarios solo puede ser 'relevance' o 'time'.")
        self.carpeta_salida.mkdir(parents=True, exist_ok=True)

## 2. Interfaz de Consola

La clase `ConsolaCarga` proporciona una capa de interfaz visual para el usuario. Utiliza la librería `rich` si está disponible para una salida más formateada y colorida; de lo contrario, recurre a `print` estándar. Esto permite una experiencia de usuario mejorada durante la ejecución del scraper.

In [ ]:
class ConsolaCarga:
    def __init__(self) -> None:
        self.rich = RICH_DISPONIBLE
        self.console = Console() if RICH_DISPONIBLE else None

    def titulo(self, texto: str, subtitulo: str = "") -> None:
        if self.rich:
            contenido = texto if not subtitulo else f"[bold]{texto}[/bold]\n{subtitulo}"
            self.console.print(Panel(contenido, title="YouTube Scraper", border_style="cyan"))
        else:
            print("=" * 60)
            print(texto)
            if subtitulo:
                print(subtitulo)
            print("=" * 60)

    def info(self, mensaje: str) -> None:
        if self.rich:
            self.console.print(f"[cyan]➜[/cyan] {mensaje}")
        else:
            print(f"-> {mensaje}")

    def ok(self, mensaje: str) -> None:
        if self.rich:
            self.console.print(f"[green]✓[/green] {mensaje}")
        else:
            print(f"OK: {mensaje}")

    def warn(self, mensaje: str) -> None:
        if self.rich:
            self.console.print(f"[yellow]⚠[/yellow] {mensaje}")
        else:
            print(f"ADVERTENCIA: {mensaje}")

    def error(self, mensaje: str) -> None:
        if self.rich:
            self.console.print(f"[red]✗[/red] {mensaje}")
        else:
            print(f"ERROR: {mensaje}")

    def resumen_final(self, filas: List[Tuple[str, Any]]) -> None:
        if self.rich:
            tabla = Table(title="Resumen final del scrapeo", show_lines=True)
            tabla.add_column("Indicador", style="cyan")
            tabla.add_column("Valor", style="white")
            for k, v in filas:
                tabla.add_row(str(k), str(v))
            self.console.print(tabla)
        else:
            print("\nRESUMEN FINAL")
            for k, v in filas:
                print(f"{k}: {v}")

    def crear_progress(self):
        if not self.rich:
            return None
        return Progress(
            SpinnerColumn(),
            TextColumn("[progress.description]{task.description}"),
            BarColumn(),
            TextColumn("{task.completed}/{task.total}"),
            TimeElapsedColumn(),
            console=self.console,
        )

## 3. Vocabulario Político Peruano

La clase `VocabularioPoliticoPeru` almacena listas de términos clave (personajes, partidos, instituciones, temas) y patrones de exclusión (ruido, desastres no políticos) utilizados para clasificar los videos y comentarios. Estos términos son esenciales para identificar contenido relevante.

In [ ]:
class VocabularioPoliticoPeru:
    PERSONAJES_POLITICOS = [
        "Dina Boluarte", "Boluarte", "Pedro Castillo", "Castillo",
        "Keiko Fujimori", "Keiko", "Alberto Fujimori", "Fujimori", "Kenji Fujimori",
        "Rafael López Aliaga", "Lopez Aliaga", "RLA", "Porky",
        "César Acuña", "Cesar Acuna", "Acuña", "Acuna",
        "Vladimir Cerrón", "Vladimir Cerron", "Cerrón", "Cerron",
        "José Jerí", "Jose Jeri", "Jerí", "Jeri",
        "José Balcázar", "Jose Balcazar", "Balcázar", "Balcazar",
        "Roberto Sánchez", "Roberto Sanchez",
        "Jorge Nieto", "Fernando Rospigliosi", "Luis Galarreta",
        "Alejandro Soto", "Eduardo Salhuana", "José Williams", "Jose Williams",
        "María del Carmen Alva", "Maria del Carmen Alva", "Lady Camones",
        "Martha Moyano", "Patricia Chirinos", "Alejandro Cavero",
        "Sigrid Bazán", "Sigrid Bazan", "Susel Paredes", "Ruth Luque",
        "Guido Bellido", "Waldemar Cerrón", "Waldemar Cerron",
        "Guillermo Bermejo", "Jorge Montoya", "José Cueto", "Jose Cueto",
        "Hernando de Soto", "Verónika Mendoza", "Veronika Mendoza",
        "Antauro Humala", "Ollanta Humala", "Martín Vizcarra", "Martin Vizcarra",
        "Francisco Sagasti", "Pedro Pablo Kuczynski", "PPK", "Alan García", "Alan Garcia",
        "Nicanor Boluarte", "Aníbal Torres", "Anibal Torres",
        "César Hildebrandt", "Cesar Hildebrandt", "Hildebrandt",
        "Gustavo Adrianzén", "Gustavo Adrianzen", "Alberto Otárola", "Alberto Otarola",
        "Juan José Santiváñez", "Juan Jose Santivanez", "Santiváñez", "Santivanez",
        "Salvador del Solar", "Betssy Chávez", "Betssy Chavez",
    ]

    PARTIDOS_Y_MOVIMIENTOS = [
        "Fuerza Popular", "Fujimorismo", "Perú Libre", "Peru Libre",
        "Renovación Popular", "Renovacion Popular", "Avanza País", "Avanza Pais",
        "Alianza para el Progreso", "APP", "Acción Popular", "Accion Popular",
        "Somos Perú", "Somos Peru", "Juntos por el Perú", "Juntos por el Peru",
        "Partido Morado", "Podemos Perú", "Podemos Peru", "APRA",
        "Frente Amplio", "Nuevo Perú", "Nuevo Peru", "Perú Primero", "Peru Primero",
        "Perú Moderno", "Peru Moderno", "Bloque Magisterial", "Bancada Socialista",
    ]

    INSTITUCIONES_POLITICAS = [
        "Congreso", "Congresista", "Congresistas", "Parlamento", "Legislativo",
        "Ejecutivo", "Gobierno", "Presidencia", "Presidente", "Presidenta",
        "Premier", "PCM", "Consejo de Ministros", "Gabinete",
        "JNE", "Jurado Nacional de Elecciones", "ONPE", "RENIEC", "JEE",
        "Fiscalía", "Fiscalia", "Ministerio Público", "Ministerio Publico",
        "Poder Judicial", "Tribunal Constitucional", "TC", "JNJ",
        "Contraloría", "Contraloria", "Defensoría", "Defensoria",
        "Municipalidad", "Alcalde", "Alcaldesa", "Regidor", "Regidores",
        "Gobernador regional", "Gobierno regional", "Mesa Directiva",
    ]

    TEMAS_POLITICOS = [
        "política", "politica", "político", "politico", "política peruana", "politica peruana",
        "elecciones", "elección", "eleccion", "electoral", "votación", "votacion", "voto", "votos",
        "campaña", "campana", "candidato", "candidata", "segunda vuelta", "debate presidencial",
        "vacancia", "censura", "interpelación", "interpelacion", "moción", "mocion",
        "cuestión de confianza", "cuestion de confianza", "adelanto de elecciones",
        "reforma política", "reforma politica", "bicameralidad", "dictadura", "democracia",
        "corrupción", "corrupcion", "coima", "soborno", "denuncia constitucional",
        "investigación fiscal", "investigacion fiscal", "acusación constitucional", "acusacion constitucional",
        "protesta", "marcha", "paro", "manifestación", "manifestacion", "conflicto social",
        "bancada", "bancadas", "pleno", "comisión", "comision", "mesa directiva",
        "ley", "proyecto de ley", "decreto", "estado de emergencia", "crisis política", "crisis politica",
    ]

    PATRONES_EXCLUSION = [
        "deportes", "fútbol", "futbol", "liga 1", "alianza lima", "universitario",
        "farandula", "farándula", "espectáculos", "espectaculos", "magaly",
        "receta", "cocina", "horóscopo", "horoscopo", "viral", "mascota",
        "accidente de tránsito", "accidente de transito", "robo", "asalto", "sicario",
        "policial", "crimen", "homicidio", "incendio",
    ]

    PATRONES_DESASTRE_NO_POLITICO = [
        "terremoto", "sismo", "temblor", "movimiento sísmico", "movimiento sismico",
        "réplica", "replica", "magnitud", "epicentro", "hipocentro",
        "alerta sísmica", "alerta sismica", "tsunami", "maremoto",
        "desastre natural", "emergencia natural", "derrumbre", "derrumbe",
        "huaico", "huayco", "inundación", "inundacion", "lluvias intensas",
        "damnificados", "rescatistas", "rescate", "heridos", "fallecidos",
        "protección civil", "proteccion civil", "defensa civil",
    ]

    TERMINOS_POLITICOS_DEBILES = [
        "Gobierno", "Presidencia", "Presidente", "Presidenta", "Ejecutivo",
        "Premier", "Gabinete", "Consejo de Ministros", "Ministerio",
        "estado de emergencia", "ley", "decreto",
    ]

## 4. Utilidades de Texto y Regex

Esta sección contiene clases utilitarias para el procesamiento de texto, como la normalización, limpieza de comentarios, extracción de handles de YouTube y anonimización de nombres de autores. También incluye `RegexMatcher` para la compilación y búsqueda eficiente de patrones de texto.

In [ ]:
class TextoUtils:
    @staticmethod
    def ahora_str() -> str:
        return datetime.now().strftime("%Y-%m-%d %H:%M:%S")

    @staticmethod
    def normalizar(texto: Any) -> str:
        if texto is None:
            return ""
        texto = str(texto).lower()
        texto = unicodedata.normalize("NFKD", texto)
        texto = "".join(c for c in texto if not unicodedata.combining(c))
        texto = re.sub(r"\s+", " ", texto).strip()
        return texto

    @staticmethod
    def limpiar_comentario(texto: Any) -> str:
        if texto is None:
            return ""
        texto = str(texto)
        texto = re.sub(r"\s+", " ", texto).strip()
        return texto

    @staticmethod
    def extraer_handle(url_canal: str) -> Optional[str]:
        match = re.search(r"@([A-Za-z0-9_.-]+)", url_canal)
        if match:
            return "@" + match.group(1)
        return None

    @staticmethod
    def anonimizar_autor(nombre: Any) -> str:
        if nombre is None:
            nombre = "desconocido"
        codigo = hashlib.sha256(str(nombre).encode("utf-8")).hexdigest()[:10]
        return f"usuario_anon_{codigo}"


class RegexMatcher:
    def __init__(self, terminos: Iterable[str]) -> None:
        self.patrones: List[Tuple[str, re.Pattern]] = []
        for termino in terminos:
            t = TextoUtils.normalizar(termino)
            if not t:
                continue
            patron = re.compile(r"(?<!\w)" + re.escape(t) + r"(?!\w)", flags=re.IGNORECASE)
            self.patrones.append((termino, patron))

    def buscar(self, texto_normalizado: str) -> List[str]:
        encontrados = [etiqueta for etiqueta, patron in self.patrones if patron.search(texto_normalizado)]
        return sorted(set(encontrados))

## 5. Cliente de YouTube API

La clase `YouTubeAPIClient` gestiona todas las interacciones con la YouTube Data API v3. Incluye métodos para resolver handles de canal, obtener IDs de videos recientes de una playlist, recuperar metadatos de videos y extraer comentarios crudos, manejando la paginación y las pausas para evitar exceder las cuotas de la API.

In [ ]:
class YouTubeAPIClient:
    def __init__(self, config: ScraperConfig, ui: ConsolaCarga) -> None:
        self.config = config
        self.ui = ui
        self.youtube = build("youtube", "v3", developerKey=config.api_key)

    def resolver_canal_por_handle(self, handle: str) -> Optional[Dict[str, Any]]:
        try:
            items = self._channels_list(handle)
            if not items and handle.startswith("@"):
                items = self._channels_list(handle[1:])
            if not items:
                return None
            item = items[0]
            return {
                "channel_id": item.get("id"),
                "titulo_canal": item.get("snippet", {}).get("title"),
                "uploads_playlist_id": item.get("contentDetails", {})
                    .get("relatedPlaylists", {})
                    .get("uploads"),
            }
        except HttpError as e:
            self.ui.error(f"Error al resolver canal {handle}: {e}")
            return None

    def _channels_list(self, handle: str) -> List[Dict[str, Any]]:
        response = self.youtube.channels().list(
            part="snippet,contentDetails",
            forHandle=handle,
        ).execute()
        return response.get("items", [])

    def obtener_ids_recientes_playlist(self, playlist_id: str, max_candidatos: int) -> List[str]:
        videos: List[str] = []
        next_page_token: Optional[str] = None

        while len(videos) < max_candidatos:
            try:
                max_resultados = min(50, max_candidatos - len(videos))
                response = self.youtube.playlistItems().list(
                    part="snippet,contentDetails",
                    playlistId=playlist_id,
                    maxResults=max_resultados,
                    pageToken=next_page_token,
                ).execute()

                for item in response.get("items", []):
                    snippet = item.get("snippet", {})
                    content = item.get("contentDetails", {})
                    video_id = content.get("videoId") or snippet.get("resourceId", {}).get("videoId")
                    if video_id:
                        videos.append(video_id)

                next_page_token = response.get("nextPageToken")
                if not next_page_token:
                    break
                time.sleep(self.config.pausa_entre_paginas)

            except HttpError as e:
                self.ui.error(f"Error al obtener playlist {playlist_id}: {e}")
                break

        return list(dict.fromkeys(videos))

    def obtener_metadatos_videos(self, video_ids: List[str]) -> List[Dict[str, Any]]:
        metadatos: List[Dict[str, Any]] = []
        for lote in self._lotes(video_ids, 50):
            try:
                response = self.youtube.videos().list(
                    part="snippet,statistics,status",
                    id=",".join(lote),
                ).execute()

                for item in response.get("items", []):
                    snippet = item.get("snippet", {})
                    stats = item.get("statistics", {})
                    status = item.get("status", {})

                    metadatos.append({
                        "video_id": item.get("id"),
                        "url_video": f"https://www.youtube.com/watch?v={item.get('id')}",
                        "titulo_video": snippet.get("title"),
                        "descripcion_video": snippet.get("description"),
                        "tags_video": snippet.get("tags", []),
                        "categoria_video": snippet.get("categoryId"),
                        "fecha_video": snippet.get("publishedAt"),
                        "canal": snippet.get("channelTitle"),
                        "channel_id": snippet.get("channelId"),
                        "vistas_video": self._int_seguro(stats.get("viewCount")),
                        "likes_video": self._int_seguro(stats.get("likeCount")),
                        "comentarios_reportados_video": self._int_seguro(stats.get("commentCount")),
                        "privacy_status": status.get("privacyStatus"),
                        "made_for_kids": status.get("madeForKids"),
                    })

                time.sleep(self.config.pausa_entre_paginas)

            except HttpError as e:
                self.ui.error(f"Error al obtener metadatos de lote: {e}")

        return metadatos

    def extraer_comentarios_crudos(self, video_meta: Dict[str, Any]) -> List[Dict[str, Any]]:
        comentarios: List[Dict[str, Any]] = []
        next_page_token: Optional[str] = None
        max_comentarios = self.config.max_comentarios_crudos_por_video

        while len(comentarios) < max_comentarios:
            try:
                restantes = max_comentarios - len(comentarios)
                max_resultados = min(self.config.max_resultados_por_pagina, restantes)

                response = self.youtube.commentThreads().list(
                    part="snippet",
                    videoId=video_meta["video_id"],
                    maxResults=max_resultados,
                    pageToken=next_page_token,
                    textFormat="plainText",
                    order=self.config.orden_comentarios,
                ).execute()

                items = response.get("items", [])
                if not items:
                    break

                for item in items:
                    top_comment = item.get("snippet", {}).get("topLevelComment", {})
                    top = top_comment.get("snippet", {})
                    texto_limpio = TextoUtils.limpiar_comentario(top.get("textDisplay"))
                    autor_original = top.get("authorDisplayName")

                    comentarios.append({
                        "video_id": video_meta.get("video_id"),
                        "url_video": video_meta.get("url_video"),
                        "titulo_video": video_meta.get("titulo_video"),
                        "medio": video_meta.get("medio"),
                        "canal": video_meta.get("canal"),
                        "fecha_video": video_meta.get("fecha_video"),
                        "vistas_video": video_meta.get("vistas_video"),
                        "likes_video": video_meta.get("likes_video"),
                        "comentarios_reportados_video": video_meta.get("comentarios_reportados_video"),
                        "categoria_video": video_meta.get("categoria_video"),
                        "puntaje_politico": video_meta.get("puntaje_politico"),
                        "razones_politicas": video_meta.get("razones_politicas"),
                        "motivo_rechazo_politico": video_meta.get("motivo_rechazo_politico"),
                        "desastre_detectado": video_meta.get("desastre_detectado"),
                        "personajes_detectados": video_meta.get("personajes_detectados"),
                        "partidos_detectados": video_meta.get("partidos_detectados"),
                        "instituciones_detectadas": video_meta.get("instituciones_detectadas"),
                        "temas_detectados": video_meta.get("temas_detectados"),
                        "comentario_id": top_comment.get("id"),
                        "autor_anonimo": TextoUtils.anonimizar_autor(autor_original),
                        "texto_comentario": texto_limpio,
                        "likes_comentario": self._int_seguro(top.get("likeCount")),
                        "fecha_comentario": top.get("publishedAt"),
                        "total_respuestas": self._int_seguro(item.get("snippet", {}).get("totalReplyCount")),
                        "fuente": "YouTube",
                        "metodo_extraccion": "YouTube Data API v3",
                        "fecha_extraccion": TextoUtils.ahora_str(),
                    })

                next_page_token = response.get("nextPageToken")
                if not next_page_token:
                    break
                time.sleep(self.config.pausa_entre_paginas)

            except HttpError as e:
                self.ui.warn(f"No se pudieron extraer comentarios de {video_meta.get('video_id')}: {e}")
                break

        return comentarios

    @staticmethod
    def _lotes(lista: List[str], n: int) -> Iterable[List[str]]:
        for i in range(0, len(lista), n):
            yield lista[i:i + n]

    @staticmethod
    def _int_seguro(valor: Any) -> Optional[int]:
        try:
            if valor is None:
                return None
            return int(valor)
        except Exception:
            return None

## 6. Clasificador Político

La clase `ClasificadorPolitico` determina si un video es de naturaleza política peruana basándose en un sistema de puntuación. Utiliza los vocabularios definidos para buscar personajes, partidos, instituciones y temas políticos en el título, descripción y tags de los videos, aplicando reglas para asignar puntajes y filtros estrictos para evitar contenido no político.

In [ ]:
class ClasificadorPolitico:
    def __init__(self, config: ScraperConfig) -> None:
        self.config = config
        self.m_personajes = RegexMatcher(VocabularioPoliticoPeru.PERSONAJES_POLITICOS)
        self.m_partidos = RegexMatcher(VocabularioPoliticoPeru.PARTIDOS_Y_MOVIMIENTOS)
        self.m_instituciones = RegexMatcher(VocabularioPoliticoPeru.INSTITUCIONES_POLITICAS)
        self.m_temas = RegexMatcher(VocabularioPoliticoPeru.TEMAS_POLITICOS)
        self.m_ruido = RegexMatcher(VocabularioPoliticoPeru.PATRONES_EXCLUSION)
        self.m_desastre = RegexMatcher(VocabularioPoliticoPeru.PATRONES_DESASTRE_NO_POLITICO)
        self.terminos_debiles_norm = {
            TextoUtils.normalizar(t) for t in VocabularioPoliticoPeru.TERMINOS_POLITICOS_DEBILES
        }

    def evaluar(self, meta: Dict[str, Any]) -> Dict[str, Any]:
        titulo = TextoUtils.normalizar(meta.get("titulo_video"))
        descripcion = TextoUtils.normalizar(meta.get("descripcion_video"))
        tags_texto = TextoUtils.normalizar(" ".join(meta.get("tags_video") or []))

        encontrados = self._detectar_en_campos(titulo, descripcion, tags_texto)
        puntaje, razones = self._calcular_puntaje(meta, encontrados)

        hay_senal_fuerte = self._hay_senal_fuerte(encontrados)
        es_politico_base = puntaje >= self.config.puntaje_minimo_politico and hay_senal_fuerte

        pasa_filtro_estricto = True
        motivo_rechazo = ""
        if self.config.filtro_meramente_politico:
            pasa_filtro_estricto, motivo_rechazo = self._validar_meramente_politico(encontrados)
            if not pasa_filtro_estricto:
                razones.append(f"rechazo por filtro estricto: {motivo_rechazo}")

        es_politico = es_politico_base and pasa_filtro_estricto

        return {
            "es_politico": es_politico,
            "puntaje_politico": puntaje,
            "razones_politicas": " | ".join(razones),
            "motivo_rechazo_politico": motivo_rechazo,
            "personajes_detectados": self._unir(encontrados, "personajes"),
            "partidos_detectados": self._unir(encontrados, "partidos"),
            "instituciones_detectadas": self._unir(encontrados, "instituciones"),
            "temas_detectados": self._unir(encontrados, "temas"),
            "ruido_detectado": self._unir(encontrados, "ruido"),
            "desastre_detectado": self._unir(encontrados, "desastre"),
        }

    def _detectar_en_campos(self, titulo: str, descripcion: str, tags: str) -> Dict[str, List[str]]:
        return {
            "personajes_titulo": self.m_personajes.buscar(titulo),
            "personajes_descripcion": self.m_personajes.buscar(descripcion),
            "personajes_tags": self.m_personajes.buscar(tags),
            "partidos_titulo": self.m_partidos.buscar(titulo),
            "partidos_descripcion": self.m_partidos.buscar(descripcion),
            "partidos_tags": self.m_partidos.buscar(tags),
            "instituciones_titulo": self.m_instituciones.buscar(titulo),
            "instituciones_descripcion": self.m_instituciones.buscar(descripcion),
            "instituciones_tags": self.m_instituciones.buscar(tags),
            "temas_titulo": self.m_temas.buscar(titulo),
            "temas_descripcion": self.m_temas.buscar(descripcion),
            "temas_tags": self.m_temas.buscar(tags),
            "ruido_titulo": self.m_ruido.buscar(titulo),
            "ruido_descripcion": self.m_ruido.buscar(descripcion),
            "ruido_tags": self.m_ruido.buscar(tags),
            "desastre_titulo": self.m_desastre.buscar(titulo),
            "desastre_descripcion": self.m_desastre.buscar(descripcion),
            "desastre_tags": self.m_desastre.buscar(tags),
        }

    def _calcular_puntaje(self, meta: Dict[str, Any], e: Dict[str, List[str]]) -> Tuple[int, List[str]]:
        puntaje = 0
        razones: List[str] = []

        reglas = [
            ("personajes_tags", 7, "tags con personaje político"),
            ("personajes_titulo", 6, "título con personaje político"),
            ("personajes_descripcion", 2, "descripción con personaje político"),
            ("partidos_tags", 6, "tags con partido/movimiento político"),
            ("partidos_titulo", 5, "título con partido/movimiento político"),
            ("partidos_descripcion", 2, "descripción con partido/movimiento político"),
            ("instituciones_tags", 5, "tags con institución política"),
            ("instituciones_titulo", 4, "título con institución política"),
            ("instituciones_descripcion", 1, "descripción con institución política"),
            ("temas_tags", 5, "tags con tema político"),
            ("temas_titulo", 4, "título con tema político"),
            ("temas_descripcion", 1, "descripción con tema político"),
        ]

        for clave, peso, razon in reglas:
            n = len(e[clave])
            if n:
                suma = peso * min(n, 2)
                puntaje += suma
                razones.append(f"{razon}: {', '.join(e[clave][:5])}")

        if str(meta.get("categoria_video")) == "25":
            puntaje += 1
            razones.append("categoría YouTube 25")

        ruido_fuerte = len(e["ruido_titulo"]) + len(e["ruido_tags"])
        ruido_debil = len(e["ruido_descripcion"])
        hay_senal_fuerte = self._hay_senal_fuerte(e)

        if ruido_fuerte and not hay_senal_fuerte:
            puntaje -= 5
            razones.append("penalización por ruido no político en título/tags")
        elif ruido_debil and not hay_senal_fuerte:
            puntaje -= 2
            razones.append("penalización por ruido no político en descripción")

        return puntaje, razones

    def _validar_meramente_politico(self, e: Dict[str, List[str]]) -> Tuple[bool, str]:
        desastre_principal = len(e["desastre_titulo"]) + len(e["desastre_tags"])
        desastre_descripcion = len(e["desastre_descripcion"])

        if self.config.bloquear_noticias_desastre and desastre_principal > 0:
            return False, "noticia de desastre/sismo/terremoto en título o tags"

        senal_actor_partido = any(len(e[k]) > 0 for k in [
            "personajes_titulo", "personajes_tags",
            "partidos_titulo", "partidos_tags",
        ])

        instituciones_principales = self._filtrar_instituciones_debiles(
            e["instituciones_titulo"] + e["instituciones_tags"]
        )
        temas_principales = e["temas_titulo"] + e["temas_tags"]

        if senal_actor_partido:
            if desastre_descripcion > 0 and not temas_principales:
                return False, "menciona desastre en descripción y no hay tema político principal"
            return True, ""

        if instituciones_principales and temas_principales:
            return True, ""

        if len(set(temas_principales)) >= 2:
            return True, ""

        if instituciones_principales and not (e["desastre_titulo"] or e["desastre_tags"]):
            return True, ""

        return False, "solo hay señales genéricas o débiles; no es meramente político"

    def _filtrar_instituciones_debiles(self, instituciones: List[str]) -> List[str]:
        fuertes = []
        for inst in instituciones:
            if TextoUtils.normalizar(inst) not in self.terminos_debiles_norm:
                fuertes.append(inst)
        return sorted(set(fuertes))

    @staticmethod
    def _hay_senal_fuerte(e: Dict[str, List[str]]) -> bool:
        claves_fuertes = [
            "personajes_tags", "personajes_titulo",
            "partidos_tags", "partidos_titulo",
            "instituciones_tags", "instituciones_titulo",
            "temas_tags", "temas_titulo",
        ]
        return any(len(e[k]) > 0 for k in claves_fuertes)

    @staticmethod
    def _unir(e: Dict[str, List[str]], prefijo: str) -> str:
        valores: List[str] = []
        for k, v in e.items():
            if k.startswith(prefijo + "_"):
                valores.extend(v)
        return "; ".join(sorted(set(valores)))

## 7. Depurador de Comentarios

La clase `DepuradorComentarios` se encarga de limpiar y filtrar los comentarios extraídos. Remueve comentarios que no cumplen con una longitud mínima, elimina duplicados y normaliza el texto para asegurar la calidad de los datos finales.

In [ ]:
class DepuradorComentarios:
    def __init__(self, config: ScraperConfig) -> None:
        self.config = config

    def depurar(self, comentarios: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
        vistos = set()
        depurados: List[Dict[str, Any]] = []

        for c in comentarios:
            texto = TextoUtils.limpiar_comentario(c.get("texto_comentario"))
            texto_norm = TextoUtils.normalizar(texto)

            if len(texto) < self.config.min_caracteres_comentario:
                continue
            if not texto_norm:
                continue

            clave = (c.get("video_id"), texto_norm)
            if clave in vistos:
                continue

            vistos.add(clave)
            c["texto_comentario"] = texto
            depurados.append(c)

        return depurados

## 8. Exportador CSV

La clase `ExportadorCSV` es responsable de guardar los comentarios depurados y el resumen del scrapeo en archivos CSV. Asegura que los datos estén en un formato accesible para análisis posteriores.

In [ ]:
class ExportadorCSV:
    def __init__(self, config: ScraperConfig) -> None:
        self.config = config

    def exportar(self, comentarios: List[Dict[str, Any]], resumen: pd.DataFrame) -> Tuple[pd.DataFrame, pd.DataFrame]:
        df_data = pd.DataFrame(comentarios)
        if not df_data.empty:
            df_data.insert(0, "n_fila", range(1, len(df_data) + 1))

        df_data.to_csv(self.config.ruta_data, index=False, encoding="utf-8-sig")
        resumen.to_csv(self.config.ruta_resumen, index=False, encoding="utf-8-sig")
        return df_data, resumen

## 9. Pipeline Principal POO

La clase `YouTubePoliticalScraperPipeline` orquesta todo el proceso de extracción. Desde la configuración inicial hasta la recolección de candidatos políticos, la extracción y selección de comentarios, y la generación del resumen final. Esta clase integra todas las funcionalidades en un flujo de trabajo cohesivo.

In [ ]:
class YouTubePoliticalScraperPipeline:
    def __init__(self, config: ScraperConfig) -> None:
        self.config = config
        self.ui = ConsolaCarga()
        self.config.validar()
        self.api = YouTubeAPIClient(config, self.ui)
        self.clasificador = ClasificadorPolitico(config)
        self.depurador = DepuradorComentarios(config)
        self.exportador = ExportadorCSV(config)

    def ejecutar(self) -> Tuple[pd.DataFrame, pd.DataFrame]:
        self.ui.titulo(
            "EXTRACCIÓN MERAMENTE POLÍTICA DE YOUTUBE",
            f"Meta: {self.config.objetivo_videos} videos × "
            f"{self.config.objetivo_comentarios_por_video} comentarios = "
            f"{self.config.meta_total_comentarios} comentarios",
        )

        candidatos, resumen_canales = self.recolectar_candidatos_politicos()
        self.ui.info(f"Candidatos políticos a intentar: {len(candidatos)}")

        comentarios_finales, resumen_videos = self.extraer_y_seleccionar_comentarios(candidatos)
        df_resumen = self.construir_resumen(resumen_canales, resumen_videos, len(candidatos))
        df_data, df_resumen = self.exportador.exportar(comentarios_finales, df_resumen)

        self.ui.resumen_final([
            ("CSV data", self.config.ruta_data),
            ("CSV resumen", self.config.ruta_resumen),
            ("Videos incluidos", df_data["video_id"].nunique() if not df_data.empty else 0),
            ("Comentarios exportados", f"{len(df_data)} / {self.config.meta_total_comentarios}"),
            ("Estado", "meta lograda" if len(df_data) >= self.config.meta_total_comentarios else "meta no lograda"),
        ])
        return df_data, df_resumen

    def recolectar_candidatos_politicos(self) -> Tuple[List[Dict[str, Any]], List[Dict[str, Any]]]:
        candidatos: List[Dict[str, Any]] = []
        resumen_canales: List[Dict[str, Any]] = []

        progress = self.ui.crear_progress()
        if progress:
            with progress:
                task = progress.add_task("Revisando canales", total=len(self.config.fuentes_canales))
                for fuente in self.config.fuentes_canales:
                    self._procesar_canal(fuente, candidatos, resumen_canales)
                    progress.advance(task)
        else:
            for fuente in self.config.fuentes_canales:
                self._procesar_canal(fuente, candidatos, resumen_canales)

        candidatos = self._ordenar_candidatos(candidatos)
        return candidatos[: self.config.max_videos_politicos_a_intentar], resumen_canales

    def _procesar_canal(self, fuente: CanalFuente, candidatos: List[Dict[str, Any]], resumen_canales: List[Dict[str, Any]]) -> None:
        handle = TextoUtils.extraer_handle(fuente.url_canal)
        self.ui.info(f"Canal: {fuente.medio} | handle: {handle}")

        if handle is None:
            resumen_canales.append(self._resumen_canal(fuente.medio, "error", "No se pudo extraer handle", 0, 0))
            return

        canal_info = self.api.resolver_canal_por_handle(handle)
        if canal_info is None or canal_info.get("uploads_playlist_id") is None:
            resumen_canales.append(self._resumen_canal(fuente.medio, "error", "No se resolvió canal o playlist", 0, 0))
            return

        ids = self.api.obtener_ids_recientes_playlist(
            canal_info["uploads_playlist_id"],
            self.config.max_candidatos_recientes_por_canal,
        )
        metas = self.api.obtener_metadatos_videos(ids)

        politicos = 0
        for meta in metas:
            meta["medio"] = fuente.medio
            evaluacion = self.clasificador.evaluar(meta)
            meta.update(evaluacion)
            if meta.get("es_politico"):
                politicos += 1
                candidatos.append(meta)

        resumen_canales.append(self._resumen_canal(fuente.medio, "ok", "Canal procesado", len(ids), politicos))
        self.ui.ok(f"{fuente.medio}: {politicos} videos políticos detectados de {len(ids)} revisados")
        time.sleep(self.config.pausa_entre_canales)

    def extraer_y_seleccionar_comentarios(self, candidatos: List[Dict[str, Any]]) -> Tuple[List[Dict[str, Any]], List[Dict[str, Any]]]:
        comentarios_finales: List[Dict[str, Any]] = []
        resumen_exitosos: List[Dict[str, Any]] = []
        reserva_parcial: List[Dict[str, Any]] = []
        videos_usados = set()

        progress = self.ui.crear_progress()
        if progress:
            with progress:
                task = progress.add_task("Extrayendo comentarios", total=len(candidatos))
                for meta in candidatos:
                    if len(resumen_exitosos) >= self.config.objetivo_videos:
                        break
                    self._intentar_video(meta, comentarios_finales, resumen_exitosos, reserva_parcial, videos_usados)
                    progress.advance(task)
        else:
            for meta in candidatos:
                if len(resumen_exitosos) >= self.config.objetivo_videos:
                    break
                self._intentar_video(meta, comentarios_finales, resumen_exitosos, reserva_parcial, videos_usados)

        resumen_final = list(resumen_exitosos)
        if len(resumen_final) < self.config.objetivo_videos:
            self.ui.warn("No se llegó a 10 videos con 50 comentarios. Se completará con parciales disponibles.")
            self._completar_con_parciales(reserva_parcial, comentarios_finales, resumen_final, videos_usados)

        return comentarios_finales, resumen_final

    def _intentar_video(
        self,
        meta: Dict[str, Any],
        comentarios_finales: List[Dict[str, Any]],
        resumen_exitosos: List[Dict[str, Any]],
        reserva_parcial: List[Dict[str, Any]],
        videos_usados: set,
    ) -> None:
        video_id = meta.get("video_id")
        if not video_id or video_id in videos_usados:
            return

        self.ui.info(f"Intentando: {video_id} | {meta.get('titulo_video')}")
        crudos = self.api.extraer_comentarios_crudos(meta)
        depurados = self.depurador.depurar(crudos)

        if len(depurados) >= self.config.objetivo_comentarios_por_video:
            seleccionados = depurados[: self.config.objetivo_comentarios_por_video]
            comentarios_finales.extend(seleccionados)
            videos_usados.add(video_id)
            resumen_exitosos.append(self._resumen_video(meta, crudos, depurados, seleccionados, "si", "exportado con meta cumplida"))
            self.ui.ok(f"{video_id}: 50 comentarios exportados")
        else:
            reserva_parcial.append({"meta": meta, "crudos": crudos, "depurados": depurados})
            self.ui.warn(f"{video_id}: solo {len(depurados)} comentarios válidos; pasa a reserva")

        time.sleep(self.config.pausa_entre_videos)

    def _completar_con_parciales(
        self,
        reserva_parcial: List[Dict[str, Any]],
        comentarios_finales: List[Dict[str, Any]],
        resumen_final: List[Dict[str, Any]],
        videos_usados: set,
    ) -> None:
        reserva_parcial = sorted(reserva_parcial, key=lambda x: len(x["depurados"]), reverse=True)

        for item in reserva_parcial:
            if len(resumen_final) >= self.config.objetivo_videos:
                break

            meta = item["meta"]
            video_id = meta.get("video_id")
            if video_id in videos_usados:
                continue

            depurados = item["depurados"]
            crudos = item["crudos"]
            seleccionados = depurados[: self.config.objetivo_comentarios_por_video]
            if not seleccionados:
                continue

            comentarios_finales.extend(seleccionados)
            videos_usados.add(video_id)
            resumen_final.append(self._resumen_video(meta, crudos, depurados, seleccionados, "no", "exportado parcial"))

    def construir_resumen(
        self,
        resumen_canales: List[Dict[str, Any]],
        resumen_videos: List[Dict[str, Any]],
        total_candidatos_detectados: int,
    ) -> pd.DataFrame:
        total_videos_exportados = len([r for r in resumen_videos if r.get("incluido_en_data") == "si"])
        total_comentarios_exportados = sum(int(r.get("comentarios_exportados") or 0) for r in resumen_videos)
        videos_con_meta = len([
            r for r in resumen_videos
            if int(r.get("comentarios_exportados") or 0) >= self.config.objetivo_comentarios_por_video
        ])
        porcentaje = round(100 * total_comentarios_exportados / self.config.meta_total_comentarios, 2) if self.config.meta_total_comentarios else 0
        estado_global = "meta lograda" if total_comentarios_exportados >= self.config.meta_total_comentarios else "meta no lograda"

        filas: List[Dict[str, Any]] = [self._fila_global(
            estado_global,
            resumen_videos,
            total_videos_exportados,
            total_comentarios_exportados,
            videos_con_meta,
            porcentaje,
            total_candidatos_detectados,
        )]

        for canal in resumen_canales:
            filas.append(self._fila_canal(canal))

        for video in resumen_videos:
            filas.append(self._fila_video(video))

        return pd.DataFrame(filas)

    def _ordenar_candidatos(self, candidatos: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
        return sorted(
            candidatos,
            key=lambda x: (
                str(x.get("fecha_video") or ""),
                int(x.get("puntaje_politico") or 0),
                int(x.get("comentarios_reportados_video") or 0),
            ),
            reverse=True,
        )

    @staticmethod
    def _resumen_canal(medio: str, estado: str, observacion: str, revisados: int, politicos: int) -> Dict[str, Any]:
        return {
            "medio": medio,
            "estado_canal": estado,
            "observacion": observacion,
            "videos_recientes_revisados": revisados,
            "videos_politicos_detectados": politicos,
        }

    def _resumen_video(
        self,
        meta: Dict[str, Any],
        crudos: List[Dict[str, Any]],
        depurados: List[Dict[str, Any]],
        seleccionados: List[Dict[str, Any]],
        cumple: str,
        estado: str,
    ) -> Dict[str, Any]:
        return {
            **meta,
            "comentarios_crudos_extraidos": len(crudos),
            "comentarios_validos_depurados": len(depurados),
            "comentarios_exportados": len(seleccionados),
            "cumple_50_comentarios": cumple,
            "incluido_en_data": "si",
            "estado": estado,
            "observacion": (
                "Video político incluido con 50 comentarios depurados."
                if cumple == "si"
                else "Video político incluido como fallback parcial por falta de videos con 50 comentarios."
            ),
        }

    def _fila_global(
        self,
        estado_global: str,
        resumen_videos: List[Dict[str, Any]],
        total_videos_exportados: int,
        total_comentarios_exportados: int,
        videos_con_meta: int,
        porcentaje: float,
        total_candidatos_detectados: int,
    ) -> Dict[str, Any]:
        return {
            "fecha_extraccion": TextoUtils.ahora_str(),
            "nivel": "GLOBAL",
            "estado": estado_global,
            "medio": "TODOS",
            "canal": "TODOS",
            "video_id": None,
            "url_video": None,
            "titulo_video": None,
            "fecha_video": None,
            "puntaje_politico": None,
            "razones_politicas": None,
            "motivo_rechazo_politico": None,
            "desastre_detectado": None,
            "personajes_detectados": None,
            "partidos_detectados": None,
            "instituciones_detectadas": None,
            "temas_detectados": None,
            "comentarios_crudos_extraidos": sum(int(r.get("comentarios_crudos_extraidos") or 0) for r in resumen_videos),
            "comentarios_validos_depurados": sum(int(r.get("comentarios_validos_depurados") or 0) for r in resumen_videos),
            "comentarios_exportados": total_comentarios_exportados,
            "cumple_50_comentarios": "si" if videos_con_meta >= self.config.objetivo_videos else "no",
            "incluido_en_data": "resumen",
            "observacion": (
                f"Videos exportados: {total_videos_exportados}/{self.config.objetivo_videos}. "
                f"Videos con {self.config.objetivo_comentarios_por_video} comentarios: "
                f"{videos_con_meta}/{self.config.objetivo_videos}. "
                f"Comentarios exportados: {total_comentarios_exportados}/{self.config.meta_total_comentarios}. "
                f"Cumplimiento: {porcentaje}%. "
                f"Candidatos políticos detectados: {total_candidatos_detectados}."
            ),
            "videos_objetivo": self.config.objetivo_videos,
            "comentarios_objetivo_por_video": self.config.objetivo_comentarios_por_video,
            "meta_total_comentarios": self.config.meta_total_comentarios,
            "porcentaje_cumplimiento": porcentaje,
        }

    def _fila_canal(self, canal: Dict[str, Any]) -> Dict[str, Any]:
        return {
            "fecha_extraccion": TextoUtils.ahora_str(),
            "nivel": "CANAL",
            "estado": canal.get("estado_canal"),
            "medio": canal.get("medio"),
            "canal": canal.get("medio"),
            "video_id": None,
            "url_video": None,
            "titulo_video": None,
            "fecha_video": None,
            "puntaje_politico": None,
            "razones_politicas": None,
            "motivo_rechazo_politico": None,
            "desastre_detectado": None,
            "personajes_detectados": None,
            "partidos_detectados": None,
            "instituciones_detectadas": None,
            "temas_detectados": None,
            "comentarios_crudos_extraidos": None,
            "comentarios_validos_depurados": None,
            "comentarios_exportados": None,
            "cumple_50_comentarios": None,
            "incluido_en_data": "no aplica",
            "observacion": (
                f"{canal.get('observacion')}. "
                f"Videos recientes revisados: {canal.get('videos_recientes_revisados')}. "
                f"Videos políticos detectados: {canal.get('videos_politicos_detectados')}."
            ),
            "videos_objetivo": self.config.objetivo_videos,
            "comentarios_objetivo_por_video": self.config.objetivo_comentarios_por_video,
            "meta_total_comentarios": self.config.meta_total_comentarios,
            "porcentaje_cumplimiento": None,
        }

    def _fila_video(self, r: Dict[str, Any]) -> Dict[str, Any]:
        return {
            "fecha_extraccion": TextoUtils.ahora_str(),
            "nivel": "VIDEO",
            "estado": r.get("estado"),
            "medio": r.get("medio"),
            "canal": r.get("canal"),
            "video_id": r.get("video_id"),
            "url_video": r.get("url_video"),
            "titulo_video": r.get("titulo_video"),
            "fecha_video": r.get("fecha_video"),
            "puntaje_politico": r.get("puntaje_politico"),
            "razones_politicas": r.get("razones_politicas"),
            "motivo_rechazo_politico": r.get("motivo_rechazo_politico"),
            "desastre_detectado": r.get("desastre_detectado"),
            "personajes_detectados": r.get("personajes_detectados"),
            "partidos_detectados": r.get("partidos_detectados"),
            "instituciones_detectadas": r.get("instituciones_detectadas"),
            "temas_detectados": r.get("temas_detectados"),
            "comentarios_crudos_extraidos": r.get("comentarios_crudos_extraidos"),
            "comentarios_validos_depurados": r.get("comentarios_validos_depurados"),
            "comentarios_exportados": r.get("comentarios_exportados"),
            "cumple_50_comentarios": r.get("cumple_50_comentarios"),
            "incluido_en_data": r.get("incluido_en_data"),
            "observacion": r.get("observacion"),
            "videos_objetivo": self.config.objetivo_videos,
            "comentarios_objetivo_por_video": self.config.objetivo_comentarios_por_video,
            "meta_total_comentarios": self.config.meta_total_comentarios,
            "porcentaje_cumplimiento": None,
        }

## 10. Ejecución

Para ejecutar el scraper, se crea una instancia de `ScraperConfig` y se inicializa `YouTubePoliticalScraperPipeline` con esta configuración. Luego, se llama al método `ejecutar()` para iniciar el proceso de extracción y guardado de datos.

In [ ]:
def main() -> None:
    config = ScraperConfig()
    pipeline = YouTubePoliticalScraperPipeline(config)
    pipeline.ejecutar()


if __name__ == "__main__":
    main()